# Simulate an LLM A/B Test

Companion notebook for the [LLM A/B Testing and Experimentation](https://codemia.ai/courses/applied_ai_systems/llm_ab_testing_and_experimentation) lesson on Codemia.

**What you will build**: a complete A/B test on simulated implicit feedback: two variants, a skewed rating distribution, a Mann-Whitney U test written from scratch, and the power curve recovered by simulation rather than looked up in a table.

**Why from scratch**: the reason to simulate is that you know the ground truth. You set the effect size, so you can ask the question you can never ask in production: at this sample size, how often would my test have found the effect that is really there, and how often would it have fired when there was nothing?

**References**:
- Mann & Whitney, [On a test of whether one of two random variables is stochastically larger than the other](https://doi.org/10.1214/aoms/1177730491) (1947)
- Kohavi, Tang & Xu, *Trustworthy Online Controlled Experiments* (2020), chapters 17-19 on metric sensitivity and sequential testing

**Time**: ~40 minutes  **GPU required**: no

## Setup

Only numpy and matplotlib. The statistics are written out by hand, partly because
the arithmetic is short and partly because the tie correction is the part everyone
gets wrong and you cannot see it when it is behind a library call.

Nothing hand-rolled is taken on trust. The test gets checked against an exact
permutation test, which needs nothing but numpy and so always runs, and additionally
against scipy where scipy happens to be installed. Those checks are the proof the
implementation is right, so do not skip them.

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

rng = np.random.default_rng(0)

try:
    from scipy import stats as scipy_stats
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
print('scipy available for cross-checking:', HAVE_SCIPY)

## Concept recap

An LLM A/B test almost never produces the data a t-test wants.

What you collect is implicit feedback: a thumbs up, a copy-to-clipboard, a
regenerate, a session that continued instead of ending. Rolled up per response that
becomes a bounded ordinal score, and the distribution is lumpy and skewed. Most
responses get the top score, a few get the bottom one, and the middle is thin. The
mean of that is a real number, but it is a bad summary, and the t-test's standard
error is derived from an assumption the data is openly violating.

Mann-Whitney U asks a question that survives the skew: **if I draw one response from
A and one from B, what is the probability that B is rated higher?** It only uses the
ordering of the scores, so it does not care that the scale is lumpy, and it does not
care that the top is clipped.

We will build the skewed generator first, so that every later result is being
measured against a distribution we can see.

## A generator with a known effect

Each variant has a latent quality. A rater converts latent quality into a 1-5 score
through a noisy, saturating function, which is what produces the clipping at 5 that
makes the distribution skewed.

`delta` is the ground truth. It is the thing a real experiment is trying to recover
and never gets to see.

In [ ]:
def draw_ratings(n, delta=0.0, rng=rng):
    # Latent helpfulness, plus per-response rater noise. The +1.1 offset puts the
    # bulk of the mass against the top of the scale, which is what production
    # implicit-feedback distributions actually look like: mostly 5s, a tail of 1s.
    latent = rng.normal(1.1 + delta, 1.0, size=n)
    scores = np.clip(np.round(3.0 + latent), 1, 5)
    return scores.astype(int)


a = draw_ratings(4000, delta=0.0)
b = draw_ratings(4000, delta=0.35)
for name, s in (('A', a), ('B', b)):
    hist = np.bincount(s, minlength=6)[1:]
    print(f'{name}: mean={s.mean():.3f}  counts 1..5 = {hist.tolist()}  '
          f'share at ceiling = {(s == 5).mean():.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
w = 0.38
ax.bar(np.arange(1, 6) - w / 2, np.bincount(a, minlength=6)[1:] / len(a), w, label='A')
ax.bar(np.arange(1, 6) + w / 2, np.bincount(b, minlength=6)[1:] / len(b), w, label='B')
ax.set_xlabel('implicit feedback score')
ax.set_ylabel('share of responses')
ax.set_title('Both variants pile up at the ceiling')
ax.legend()
plt.tight_layout()
plt.show()

That shape is the whole argument. The two variants differ, but they differ in
*where the mass sits*, not in a tidy shift of a bell curve. Nearly half of each
sample is the same clipped value of 5, which means the data is mostly ties, and ties
are exactly what the textbook U statistic mishandles unless you correct for them.

## Mann-Whitney U, from scratch

Three pieces: average ranks over the pooled sample, the U statistic, and a normal
approximation with both a tie correction and a continuity correction.

The tie correction matters here more than in most datasets. With half the sample
sitting on one value, the uncorrected variance is too large, the z too small, and the
test too conservative. Skipping it does not break the test, it quietly costs you
power, which is the kind of bug that never announces itself.

In [ ]:
def rankdata(x):
    # Average ranks, vectorised. Assign 1..N in sorted order, then replace every
    # tied group with the mean of the ranks it occupies.
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind='mergesort')
    raw = np.empty(len(x), dtype=float)
    raw[order] = np.arange(1, len(x) + 1)
    _, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
    group_sums = np.bincount(inv, weights=raw)
    return (group_sums / counts)[inv], counts

In [ ]:
def mann_whitney(x, y):
    # Returns (U for x, z, two-sided p, rank-biserial effect size).
    n1, n2 = len(x), len(y)
    ranks, tie_counts = rankdata(np.concatenate([x, y]))
    r1 = ranks[:n1].sum()
    u1 = r1 - n1 * (n1 + 1) / 2.0            # U counts x-beats-y pairs, ties as half
    mu = n1 * n2 / 2.0
    n = n1 + n2
    # Tie correction: subtract the rank mass consumed by tied groups.
    tie_term = (tie_counts ** 3 - tie_counts).sum() / (n * (n - 1))
    var = n1 * n2 / 12.0 * ((n + 1) - tie_term)
    if var <= 0:
        return u1, 0.0, 1.0, 0.0
    # Continuity correction pulls U half a step toward the null before standardising.
    z = (u1 - mu - math.copysign(0.5, u1 - mu)) / math.sqrt(var)
    p = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(abs(z) / math.sqrt(2.0))))
    # Rank-biserial r = 2*A - 1, where A = P(y > x) + 0.5*P(tie): a real effect size,
    # unlike p, which is mostly a statement about sample size.
    a_common = u1 / (n1 * n2)
    return u1, z, p, 2.0 * a_common - 1.0

In [ ]:
# The cross-check. If this disagrees with scipy, the implementation above is wrong
# and every number later in the notebook is wrong with it.
x, y = draw_ratings(300, 0.0), draw_ratings(300, 0.35)
u, z, p, rb = mann_whitney(x, y)
print(f'ours : U={u:.1f}  z={z:+.4f}  p={p:.6f}  rank-biserial={rb:+.4f}')
if HAVE_SCIPY:
    res = scipy_stats.mannwhitneyu(x, y, alternative='two-sided', use_continuity=True)
    print(f'scipy: U={res.statistic:.1f}              p={res.pvalue:.6f}')
    assert abs(u - res.statistic) < 1e-6, 'U disagrees with scipy'
    assert abs(p - res.pvalue) < 2e-4, f'p disagrees with scipy: {p} vs {res.pvalue}'
    print('cross-check passed')
else:
    print('scipy absent, falling back to the permutation check below')

That cross-check only runs where scipy is installed. The one below runs everywhere,
and it is the stronger of the two: rather than comparing our normal approximation
against somebody else's normal approximation, it compares it against the exact null
distribution, built by shuffling.

In [ ]:
# Under the null, the two arms are interchangeable, so every way of dealing the
# pooled ranks into arms of size n1 and n2 is equally likely. Shuffling therefore
# builds the true null distribution of U and the p-value becomes a counting exercise,
# with no appeal to normality, no variance formula and no tie correction needed.
def mann_whitney_permutation(x, y, reps=4999, seed=0):
    r = np.random.default_rng(seed)
    n1, n2 = len(x), len(y)
    ranks, _ = rankdata(np.concatenate([x, y]))
    u_obs = ranks[:n1].sum() - n1 * (n1 + 1) / 2.0
    # One shuffle = one choice of which n1 ranks land in arm A, vectorised over reps.
    picks = np.argsort(r.random((reps, n1 + n2)), axis=1)[:, :n1]
    u_null = ranks[picks].sum(axis=1) - n1 * (n1 + 1) / 2.0
    centre = n1 * n2 / 2.0
    # The +1 on both sides counts the observed value as one of its own permutations.
    # Without it a p-value of exactly 0 is reportable, which is never true.
    return (1 + np.sum(np.abs(u_null - centre) >= abs(u_obs - centre))) / (reps + 1)


p_perm = mann_whitney_permutation(x, y)
print(f'normal approximation p = {p:.6f}')
print(f'exact permutation    p = {p_perm:.6f}   (4999 shuffles)')
# A tolerance, not an equality: these are two different estimators of the same
# quantity. The band is measured, not hoped for. Sweeping n in {50, 100, 300, 600}
# against delta in {0, 0.1, 0.2, 0.35}, the largest gap seen was 0.0122, at a p near
# 0.3 where the permutation estimate carries about 0.007 of sampling error itself.
assert abs(p - p_perm) < 0.02, f'approximation and exact disagree: {p} vs {p_perm}'
print('the normal approximation is trustworthy on this data')

## Experiment: one test, the way you would actually run it

One experiment, 400 responses per arm, a true effect of `delta=0.35`. We report the
t-test alongside, not because it is the right tool but because it is the tool most
dashboards reach for, and the gap between the two is worth seeing on real skew.

In [ ]:
def welch_t(x, y):
    # Welch's t, written out so the comparison is apples to apples.
    nx, ny = len(x), len(y)
    vx, vy = x.var(ddof=1) / nx, y.var(ddof=1) / ny
    t = (y.mean() - x.mean()) / math.sqrt(vx + vy)
    df = (vx + vy) ** 2 / (vx ** 2 / (nx - 1) + vy ** 2 / (ny - 1))
    # Normal tail is close enough at these df and keeps the notebook scipy-free.
    p = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(abs(t) / math.sqrt(2.0))))
    return t, df, p

In [ ]:
N = 400
xa, xb = draw_ratings(N, 0.0), draw_ratings(N, 0.35)
u, z, p, rb = mann_whitney(xa, xb)
t, df, pt = welch_t(xa, xb)

print(f'n per arm            {N}')
print(f'mean A / mean B      {xa.mean():.4f} / {xb.mean():.4f}   '
      f'(lift {100 * (xb.mean() / xa.mean() - 1):+.2f} pct)')
print(f'ties at score 5      {(xa == 5).mean():.1%} / {(xb == 5).mean():.1%}')
print(f'Mann-Whitney         U={u:.0f}  z={z:+.3f}  p={p:.5f}')
print(f'rank-biserial        {rb:+.4f}  (P(B>A) corrected for ties = {(rb + 1) / 2:.4f})')
print(f'Welch t              t={t:+.3f}  df={df:.1f}  p={pt:.5f}')

## The power curve

This is the part a table cannot give you, because the table assumes a distribution
this data does not have.

For each sample size we run the whole experiment many times, with the effect really
present, and count how often the test calls it. That fraction *is* the power. Then we
do it again with `delta=0` to measure the false positive rate, which should land on
alpha and is the sanity check on everything above.

In [ ]:
def power_at(n, delta, reps=600, alpha=0.05, seed=1):
    r = np.random.default_rng(seed)
    hits = 0
    for _ in range(reps):
        xa = draw_ratings(n, 0.0, rng=r)
        xb = draw_ratings(n, delta, rng=r)
        hits += mann_whitney(xa, xb)[2] < alpha
    return hits / reps


SIZES = [25, 50, 100, 200, 400, 800]
power = [power_at(n, 0.35) for n in SIZES]
null = [power_at(n, 0.0, seed=7) for n in SIZES]
for n, pw, fp in zip(SIZES, power, null):
    print(f'n={n:4d} per arm   power(delta=0.35) = {pw:.3f}   '
          f'false positive rate(delta=0) = {fp:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(SIZES, power, 'o-', label='power at delta=0.35')
ax.plot(SIZES, null, 's--', label='false positives at delta=0')
ax.axhline(0.8, color='grey', lw=1, ls=':')
ax.axhline(0.05, color='grey', lw=1, ls=':')
ax.set_xscale('log')
ax.set_xlabel('responses per arm')
ax.set_ylabel('share of runs with p < 0.05')
ax.set_ylim(0, 1.02)
ax.set_title('Power recovered by simulation, not by table')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Variations

Three things that change the answer, in rough order of how often they bite.

In [ ]:
# 1. Effect size. Power is far more sensitive to delta than to anything you can
# fix by collecting more data, which is why a weak metric cannot be rescued by n.
for d in (0.10, 0.20, 0.35, 0.50):
    row = [power_at(n, d, reps=400, seed=3) for n in (100, 400, 1600)]
    print(f'delta={d:.2f}   n=100: {row[0]:.3f}   n=400: {row[1]:.3f}   '
          f'n=1600: {row[2]:.3f}')

In [ ]:
# 2. Peeking. Test after every 25 responses and stop the first time p < 0.05.
# Ground truth here is delta=0, so every rejection is a false positive.
def peeking_false_positive(n_max=400, step=25, reps=600, alpha=0.05, seed=11):
    r = np.random.default_rng(seed)
    fired = 0
    for _ in range(reps):
        xa, xb = draw_ratings(n_max, 0.0, rng=r), draw_ratings(n_max, 0.0, rng=r)
        for k in range(step, n_max + 1, step):
            if mann_whitney(xa[:k], xb[:k])[2] < alpha:
                fired += 1
                break
    return fired / reps


print(f'fixed-n false positive rate at n=400 : {power_at(400, 0.0, reps=600, seed=11):.3f}')
print(f'peeking every 25 up to n=400         : {peeking_false_positive():.3f}')

In [ ]:
# 3. The tie correction we were careful about. Drop it and see what it was buying.
def mann_whitney_no_tie_fix(x, y):
    n1, n2 = len(x), len(y)
    ranks, _ = rankdata(np.concatenate([x, y]))
    u1 = ranks[:n1].sum() - n1 * (n1 + 1) / 2.0
    mu, n = n1 * n2 / 2.0, n1 + n2
    var = n1 * n2 * (n + 1) / 12.0            # textbook form, assumes no ties
    z = (u1 - mu - math.copysign(0.5, u1 - mu)) / math.sqrt(var)
    return 2.0 * (1.0 - 0.5 * (1.0 + math.erf(abs(z) / math.sqrt(2.0))))


r = np.random.default_rng(5)
with_fix = without = 0
for _ in range(600):
    xa, xb = draw_ratings(200, 0.0, rng=r), draw_ratings(200, 0.35, rng=r)
    with_fix += mann_whitney(xa, xb)[2] < 0.05
    without += mann_whitney_no_tie_fix(xa, xb) < 0.05
print(f'power at n=200 with tie correction    : {with_fix / 600:.3f}')
print(f'power at n=200 without tie correction : {without / 600:.3f}')

Read the numbers above in this order.

1. **The power curve is the deliverable, not the single p-value.** A test that reaches
   0.8 power only at several hundred responses per arm tells you the experiment needs
   a week of traffic, and that is a planning fact you want before you ship the
   variant, not after an inconclusive week.
2. **The false positive line is the audit.** It should sit on 0.05 at every sample
   size, because it is measured with `delta=0`, where there is nothing to detect. Read
   it with its error bar: each rate is a proportion out of 600 simulated experiments,
   so its standard error is `sqrt(0.05 * 0.95 / 600)`, about 0.009. Anything from
   roughly 0.032 to 0.068 is consistent with a correctly calibrated test, and the
   printed values wander inside that band. A line that sat on 0.08 at every n, or
   climbed with n, would be a real miscalibration and would void the power numbers
   above it.
3. **Peeking is not a small sin.** Stopping the first time you see significance, with
   no true effect present, fires far above the nominal 5 percent. The fix is a
   sequential method with a spending function, or committing to a fixed n and not
   looking.
4. **The tie correction buys power, silently.** Both versions are valid tests; the
   uncorrected one is just weaker on data that is half ties, and nothing in its output
   says so.
5. **`p` is not an effect size.** Report the rank-biserial correlation, or
   equivalently P(B beats A), next to it. A tiny effect at huge n and a large effect
   at small n produce the same p and call for different decisions.

## What to try next

- Replace the synthetic generator with your own exported ratings and re-run the power
  curve. You will usually find the real effect sizes are smaller than the synthetic
  ones, and that the sample size you have been using is underpowered.
- Add a sequential test: implement an alpha-spending boundary (O'Brien-Fleming is the
  usual choice) and show that peeking against that boundary holds the false positive
  rate at 0.05.
- Switch the metric. Compare the power of the 1-5 rating against a binary
  thumbs-up, measured on the same latent quality. Binarising throws away information
  and the power curve shows exactly how much.
- Model the unit of randomisation properly. Ratings from the same user are
  correlated, so randomising by response inflates significance. Simulate 30 responses
  per user and watch the false positive rate climb above 0.05 even with fixed n.
- Add a guardrail metric that must not regress (latency, refusal rate) and apply a
  multiple-comparison correction across the primary and guardrail tests.
- Measure the minimum detectable effect directly: for a fixed n and 0.8 target power,
  binary-search over `delta` instead of reading power off a curve.